In [1]:
# =========================
# 1. IMPORTS + DEVICE
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# 2. DATA (IMPROVED)
# =========================
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=128, shuffle=False)

# =========================
# 3. PRUNABLE LAYER
# =========================
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features))

        self.temperature = 1.0

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores / self.temperature)
        pruned_weights = self.weight * gates
        return F.linear(x, pruned_weights, self.bias)

    def get_gates(self):
        return torch.sigmoid(self.gate_scores / self.temperature)

    def set_temperature(self, temp):
        self.temperature = temp

# =========================
# 4. MODEL (UPGRADED)
# =========================
class PrunableNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc1 = PrunableLinear(64 * 8 * 8, 512)  # increased
        self.fc2 = PrunableLinear(512, 10)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# =========================
# 5. SPARSITY LOSS (FIXED)
# =========================
def compute_sparsity_loss(model):
    total_loss = 0.0
    count = 0

    for module in model.modules():
        if hasattr(module, "get_gates"):
            gates = module.get_gates()
            total_loss += torch.mean(gates)  # normalized
            count += 1

    return total_loss / count

# =========================
# 6. SPARSITY METRIC
# =========================
def compute_sparsity(model, threshold=1e-2):
    total = 0
    zero = 0

    for module in model.modules():
        if hasattr(module, "get_gates"):
            gates = module.get_gates()
            total += gates.numel()
            zero += (gates < threshold).sum().item()

    return zero / total

# =========================
# 7. HARD PRUNING
# =========================
def hard_prune(model, threshold=0.1):
    for module in model.modules():
        if hasattr(module, "get_gates"):
            gates = module.get_gates()
            mask = (gates > threshold).float()
            module.weight.data *= mask

# =========================
# 8. TRAIN FUNCTION (UPGRADED)
# =========================
def train(model, epochs=50, base_lambda=1e-3):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        # Smooth lambda increase
        lambda_sparse = base_lambda * (epoch / epochs) ** 2

        # Strong temperature annealing
        temp = max(0.05, 1.0 * (0.95 ** epoch))

        for module in model.modules():
            if hasattr(module, "set_temperature"):
                module.set_temperature(temp)

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            ce_loss = criterion(outputs, labels)
            sparse_loss = compute_sparsity_loss(model)

            loss = ce_loss + lambda_sparse * sparse_loss

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        sparsity = compute_sparsity(model)
        print(f"Epoch {epoch+1}: Loss={running_loss:.4f}, Sparsity={sparsity:.4f}")

    return model

# =========================
# 9. EVALUATION
# =========================
def evaluate(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    sparsity = compute_sparsity(model)

    return acc, sparsity

# =========================
# 10. RUN EXPERIMENTS
# =========================
def run_experiments():
    lambdas = [1e-3, 5e-3, 1e-2]
    results = []

    for lam in lambdas:
        print(f"\nTraining with lambda = {lam}")

        model = PrunableNet()
        model = train(model, epochs=50, base_lambda=lam)

        # Apply hard pruning
        hard_prune(model)

        acc, sparsity = evaluate(model)

        results.append((lam, acc, sparsity))

    print("\nFinal Results:")
    print("Lambda\tAccuracy\tSparsity")

    for lam, acc, sparsity in results:
        print(f"{lam}\t{acc:.2f}%\t\t{sparsity:.2f}")

# =========================
# 11. RUN
# =========================
run_experiments()

Using device: cuda


100%|██████████| 170M/170M [00:05<00:00, 33.9MB/s]



Training with lambda = 0.001
Epoch 1: Loss=643.4245, Sparsity=0.0000
Epoch 2: Loss=519.7044, Sparsity=0.0000
Epoch 3: Loss=458.2497, Sparsity=0.0000
Epoch 4: Loss=414.0838, Sparsity=0.0000
Epoch 5: Loss=384.4086, Sparsity=0.0001
Epoch 6: Loss=361.9570, Sparsity=0.0002
Epoch 7: Loss=344.9422, Sparsity=0.0003
Epoch 8: Loss=327.1388, Sparsity=0.0006
Epoch 9: Loss=314.8578, Sparsity=0.0011
Epoch 10: Loss=303.6981, Sparsity=0.0019
Epoch 11: Loss=293.0110, Sparsity=0.0030
Epoch 12: Loss=282.5599, Sparsity=0.0045
Epoch 13: Loss=274.1884, Sparsity=0.0065
Epoch 14: Loss=266.6784, Sparsity=0.0092
Epoch 15: Loss=258.8861, Sparsity=0.0127
Epoch 16: Loss=251.9271, Sparsity=0.0169
Epoch 17: Loss=248.4240, Sparsity=0.0219
Epoch 18: Loss=239.5023, Sparsity=0.0278
Epoch 19: Loss=235.0103, Sparsity=0.0346
Epoch 20: Loss=230.1922, Sparsity=0.0421
Epoch 21: Loss=225.8769, Sparsity=0.0506
Epoch 22: Loss=220.8696, Sparsity=0.0598
Epoch 23: Loss=217.8371, Sparsity=0.0699
Epoch 24: Loss=214.6794, Sparsity=0.